# PyTorch Dataset and DataLoader Basics

In this notebook, We will learn:

- what a Dataset is in PyTorch
- what a DataLoader does
- how to create a custom dataset
- how batching works
- how shuffling works
- why Dataset and DataLoader are important in deep learning

## 1. Why Do We Need Dataset and DataLoader?

In deep learning, models are trained on data.

But real-world data is usually large, so we do not give the whole dataset to the model at once.

Instead, we:

- store data in a structured form
- access one sample at a time when needed
- group samples into batches
- optionally shuffle data during training

PyTorch uses:

- `Dataset` → to store and access data
- `DataLoader` → to load the data in batches

## 2. Create a Small Sample Dataset

We will create a small binary classification dataset using `make_classification`.

This is only for learning, so we keep the dataset very small.

In [1]:
from sklearn.datasets import make_classification
import torch

X, y = make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

## 3. Inspect Features and Labels

Let us look at the input features and target labels.

In [2]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (10, 2)
Shape of y: (10,)


In [3]:
print("Features (X):")
print(X)

Features (X):
[[ 1.06833894 -0.97007347]
 [-1.14021544 -0.83879234]
 [-2.8953973   1.97686236]
 [-0.72063436 -0.96059253]
 [-1.96287438 -0.99225135]
 [-0.9382051  -0.54304815]
 [ 1.72725924 -1.18582677]
 [ 1.77736657  1.51157598]
 [ 1.89969252  0.83444483]
 [-0.58723065 -1.97171753]]


In [4]:
print("Labels (y):")
print(y)

Labels (y):
[1 0 0 0 0 1 1 1 1 0]


### Reason

- `X` contains the input features
- `y` contains the target labels
- each row in `X` is one sample
- each value in `y` is the label for that sample

Since `X.shape = (10, 2)`, we have:
- 10 samples
- 2 features per sample

## 4. Why Do We Need a Custom Dataset?

Right now, `X` and `y` are just separate arrays.

But PyTorch training usually expects data in a structured format.

A custom `Dataset` helps us:

- keep features and labels together
- access one sample using an index
- easily connect our data to a `DataLoader`

This makes data handling clean and reusable.

In [5]:
from torch.utils.data import Dataset, DataLoader

## 5. Create a Custom Dataset Class

In PyTorch, a custom dataset usually defines three important methods:

- `__init__()` → stores the data
- `__len__()` → tells how many samples are in the dataset
- `__getitem__()` → returns one sample and its label

In [6]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

## 6. Understanding the Methods

### `__init__(self, features, labels)`
This stores the input features and labels inside the dataset object.

### `__len__(self)`
This returns the total number of samples in the dataset.

### `__getitem__(self, index)`
This returns one sample at a given index.

For example:
- feature row at position `index`
- label at position `index`

This is how PyTorch knows how to fetch data one sample at a time.


## 7. Create the Dataset Object

In [7]:
dataset = CustomDataset(X, y)

In [8]:
print("Length of dataset:", len(dataset))

Length of dataset: 10


### Reason

This works because `len(dataset)` internally calls:

```python
dataset.__len__()

## 8. Access Individual Samples

A dataset lets us access one sample at a time using indexing.

In [9]:
sample_feature, sample_label = dataset[2]

print("Feature at index 2:", sample_feature)
print("Label at index 2:", sample_label)

Feature at index 2: [-2.8953973   1.97686236]
Label at index 2: 0


### Reason

This works because `dataset[2]` internally calls:

```python
dataset.__getitem__(2)

In [10]:
for i in range(3):
    feature, label = dataset[i]
    print(f"Sample {i}")
    print("Feature:", feature)
    print("Label:", label)
    print()

Sample 0
Feature: [ 1.06833894 -0.97007347]
Label: 1

Sample 1
Feature: [-1.14021544 -0.83879234]
Label: 0

Sample 2
Feature: [-2.8953973   1.97686236]
Label: 0



## 9. What is a DataLoader?

A `DataLoader` takes a dataset and helps us load the data in batches.

It can also:

- shuffle the data
- iterate automatically
- make training loops easier

So:

- `Dataset` defines **how to get one sample**
- `DataLoader` defines **how to group and serve many samples**

## 10. Create a DataLoader

We will use:

- `batch_size=4`
- `shuffle=False`

This means:
- 4 samples per batch
- data will be loaded in the original order

In [11]:
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

## 11. Iterate Through the DataLoader

In [12]:
for batch_features, batch_labels in dataloader:
    print("Batch features:")
    print(batch_features)
    print("Batch labels:")
    print(batch_labels)
    print("-" * 40)

Batch features:
tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606]], dtype=torch.float64)
Batch labels:
tensor([1, 0, 0, 0])
----------------------------------------
Batch features:
tensor([[-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116]], dtype=torch.float64)
Batch labels:
tensor([0, 1, 1, 1])
----------------------------------------
Batch features:
tensor([[ 1.8997,  0.8344],
        [-0.5872, -1.9717]], dtype=torch.float64)
Batch labels:
tensor([1, 0])
----------------------------------------


## 12. What is a Batch?

A batch is a small group of samples processed together.

Since our dataset has 10 samples and `batch_size=4`:

- batch 1 → 4 samples
- batch 2 → 4 samples
- batch 3 → 2 samples

The last batch may be smaller if the dataset size is not perfectly divisible by the batch size.

In [13]:
print("Number of batches:", len(dataloader))

Number of batches: 3


### Reason

We have 10 samples and batch size is 4.

So the batches are:
- 4 samples
- 4 samples
- 2 samples

That gives a total of 3 batches.


## 13. What Does `shuffle=True` Do?

Shuffling changes the order of the samples before creating batches.

This is useful during training because:
- the model does not always see data in the same order
- training becomes less biased by sample order
- learning is often better

In [14]:
dataloader_no_shuffle = DataLoader(dataset, batch_size=4, shuffle=False)
dataloader_shuffle = DataLoader(dataset, batch_size=4, shuffle=True)

print("Without shuffle:")
for batch_features, batch_labels in dataloader_no_shuffle:
    print(batch_labels)

Without shuffle:
tensor([1, 0, 0, 0])
tensor([0, 1, 1, 1])
tensor([1, 0])


In [15]:
print("With shuffle:")
for batch_features, batch_labels in dataloader_shuffle:
    print(batch_labels)

With shuffle:
tensor([0, 1, 1, 0])
tensor([1, 1, 0, 0])
tensor([1, 0])


### Reason

When `shuffle=False`, the order stays fixed.

When `shuffle=True`, the order changes.

This is especially important during model training.

## 14. Converting Data to Tensors

So far, `X` and `y` come from scikit-learn and are NumPy arrays.

PyTorch models usually work with tensors.

So in practice, we should convert the data to tensors.

In [16]:
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

print(type(X_tensor))
print(type(y_tensor))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


## 15. Better Version of the Dataset

Now we will use tensor data instead of NumPy arrays.

In [17]:
class TensorDatasetCustom(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [18]:
tensor_dataset = TensorDatasetCustom(X, y)
tensor_dataloader = DataLoader(tensor_dataset, batch_size=4, shuffle=True)

for batch_features, batch_labels in tensor_dataloader:
    print(batch_features)
    print(batch_labels)
    print()

tensor([[ 1.7273, -1.1858],
        [-0.5872, -1.9717],
        [-1.9629, -0.9923],
        [-2.8954,  1.9769]])
tensor([1., 0., 0., 0.])

tensor([[ 1.7774,  1.5116],
        [-0.9382, -0.5430],
        [ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1., 1., 1., 0.])

tensor([[ 1.8997,  0.8344],
        [-0.7206, -0.9606]])
tensor([1., 0.])



### Reason

This version is more practical because PyTorch models expect tensor inputs.

So this is closer to what we do in real training pipelines.

## 16. Why Are Dataset and DataLoader Important in Training?

During training, the model usually runs like this:

1. load one batch from the dataloader
2. make predictions
3. calculate loss
4. compute gradients
5. update weights
6. repeat for the next batch

So the `DataLoader` is what feeds data to the model step by step.

## 17. Small Training Example

Let us see how a dataloader is used inside a training loop.

In [19]:
import torch.nn as nn
import torch.optim as optim

model = nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [20]:
for epoch in range(3):
    print(f"Epoch {epoch+1}")
    
    for batch_features, batch_labels in tensor_dataloader:
        batch_labels = batch_labels.view(-1, 1)

        predictions = model(batch_features)
        loss = criterion(predictions, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print("Batch loss:", loss.item())
    
    print("-" * 40)

Epoch 1
Batch loss: 1.2545522451400757
Batch loss: 1.5425715446472168
Batch loss: 0.41878557205200195
----------------------------------------
Epoch 2
Batch loss: 1.2372101545333862
Batch loss: 0.4513958692550659
Batch loss: 1.527223825454712
----------------------------------------
Epoch 3
Batch loss: 0.9069192409515381
Batch loss: 0.6330974698066711
Batch loss: 0.7693905830383301
----------------------------------------


## 18. What Happened in the Training Loop?

For each batch:

- the model received a small group of samples
- it made predictions
- loss was calculated
- gradients were computed
- weights were updated

This is why dataloaders are essential in deep learning.
Without them, batching and iteration would be much harder to manage manually.


## 19. Practical Notes

### Batch Size
- small batch size → less memory, noisier updates
- large batch size → more memory, smoother updates

### Shuffle
- usually `shuffle=True` for training
- usually `shuffle=False` for validation or testing

### Dataset
- useful when data needs custom handling
- can load images, text, CSV files, etc.

### DataLoader
- makes training loops clean and efficient


## 20. Summary

In this notebook, We learned:

- `Dataset` is used to store and access data
- `__len__()` returns dataset size
- `__getitem__()` returns one sample
- `DataLoader` loads data in batches
- `batch_size` controls how many samples are loaded together
- `shuffle=True` changes the order of samples
- `DataLoader` is used inside training loops to feed data batch by batch